# 2026-09-01 — 機能A のフォールバックと機能B の初回実測

APIキーが 8/31 に届いたので、これまで動かせなかった **LLM を呼ぶ部分**を初めて回した日の記録。
問いは2つだけ。

1. **機能A** — 近傍分類が自信を持てなかった行を LLM に回すと、実際に正しくなるのか
2. **機能B** — 統計モデルに先に解かせ、その予測を証拠として LLM に最終判断させると、統計モデル単体より良くなるのか

用語: **MAE**（平均絶対誤差 = 予測が平均で何万円ずれるか。小さいほど良い）、
**accuracy**（正解率）、**アブレーション**（部品を1つ外して効いていたか確かめる実験）。

詳細は [`docs/2026-09-01-feature-fallback.md`](../docs/2026-09-01-feature-fallback.md) と
[`docs/2026-09-01-llm-predictor.md`](../docs/2026-09-01-llm-predictor.md)。
このノートは `results/*.csv` を読むだけなので **API は呼ばない（無料・再実行可）**。

In [6]:
import pandas as pd, numpy as np
from pathlib import Path

R = Path("..") / "results"
read = lambda n: pd.read_csv(R / n, encoding="utf-8-sig")
pd.set_option("display.width", 200)

cost = read("llm_cost.csv")
print(f"この日までの API 費用の累計: ${cost['費用_usd'].sum():.2f}"
      f"（{len(cost):,} 回の呼び出し）")

この日までの API 費用の累計: $17.38（1,923 回の呼び出し）


## 1. 機能A のフォールバック — 成立した

確信度の低い順に何割を LLM に回すか（`escalate_rate`）を振って、
タイトル列から作ったグレード名が正規表現版とどれだけ一致するかを見る。
宣言した10値は実データの 90.5% しかカバーしないので、
**「宣言値に限った accuracy」が主指標**（残り 9.5% は正解になりようがない）。

In [3]:
fb = read("feature_fallback.csv")
print(fb.round(3).to_string(index=False))

base = fb.loc[0, "宣言値に限った accuracy"]
for _, r in fb.iloc[1:].iterrows():
    n = int(r["実際に回した行"])
    print(f"{r['エスカレーション率']:.0%} 回す → 宣言値内 accuracy "
          f"{r['宣言値に限った accuracy']:.3f}（{r['宣言値に限った accuracy']-base:+.3f}） / "
          f"{n} 行 / ${r['費用_usd']:.3f}（1行 ${r['費用_usd']/n:.4f}）")
print("\n→ LLM に回した行のうち宣言値内のものは、どの率でも 100% 正解だった。")
print("→ 15% 回すだけで改善の 3 分の 2 が取れる。全行に投げる必要はない。")

 エスカレーション率  実際に回した行  accuracy  宣言値に限った accuracy  回した行の accuracy  回した行の宣言値内 accuracy  費用_usd
      0.00        0     0.790             0.873             NaN                 NaN   0.000
      0.05       20     0.832             0.920           0.950                 1.0   0.106
      0.15       60     0.852             0.942           0.917                 1.0   0.215
      0.30      120     0.885             0.978           0.842                 1.0   0.333
5% 回す → 宣言値内 accuracy 0.920（+0.047） / 20 行 / $0.106（1行 $0.0053）
15% 回す → 宣言値内 accuracy 0.942（+0.069） / 60 行 / $0.215（1行 $0.0036）
30% 回す → 宣言値内 accuracy 0.978（+0.105） / 120 行 / $0.333（1行 $0.0028）

→ LLM に回した行のうち宣言値内のものは、どの率でも 100% 正解だった。
→ 15% 回すだけで改善の 3 分の 2 が取れる。全行に投げる必要はない。


## 2. 機能B — 統計モデルに届かず。ただし弱い証拠を外すと有意に改善

各 fold の test から 60 行ずつ、計 300 行で採点。
**統計モデルと LLM をまったく同じ行で採点している**ので比較は成り立つ（全行の 12.21 とは直接比べられない)。

- `all` … 設計書どおり証拠を全部渡す（LightGBM・XGBoost・**近傍5件の中央値**）
- `trees` … 弱い証拠（近傍中央値・単体 MAE 30.86）を外したアブレーション

In [3]:
a, t = read("llm_predictor.csv"), read("llm_predictor_trees.csv")
tbl = pd.Series({
    "機能B（証拠 = 全部・設計書どおり）": a["MAE_機能B"].mean(),
    "機能B（証拠 = 木2つだけ）":          t["MAE_機能B"].mean(),
    "LightGBM（＝超えるべき線）":         a["MAE_LightGBM"].mean(),
    "XGBoost":                            a["MAE_XGBoost"].mean(),
    "近傍5件の価格中央値":                 a["MAE_近傍5件の中央値"].mean(),
}, name="MAE（万円）")
print(tbl.round(2).to_string(), "\n")

# 行ごとのブートストラップ（データを復元抽出して差の分布を見る）で有意性を確かめる
d  = read("llm_predictor_trees_rows.csv")
y, llm, lg = (d[c].to_numpy(float) for c in ("実際", "機能B", "LightGBM"))
llm_all = read("llm_predictor_rows.csv")["機能B"].to_numpy(float)
idx = np.random.default_rng(0).integers(0, len(y), size=(2000, len(y)))

for name, diff in [("trees − all（弱い証拠を外した効果）", np.abs(llm-y) - np.abs(llm_all-y)),
                   ("trees − LightGBM（S6 の受け入れ基準）", np.abs(llm-y) - np.abs(lg-y))]:
    bs = diff[idx].mean(axis=1)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"{name}: {diff.mean():+.3f} 万円 / 95%区間 [{lo:+.3f}, {hi:+.3f}] / "
          f"良い確率 {(bs < 0).mean():.1%}"
          f"{'  ← 有意' if hi < 0 else '  ← 0 をまたぐ＝有意でない'}")

機能B（証拠 = 全部・設計書どおり）    13.59
機能B（証拠 = 木2つだけ）        12.85
LightGBM（＝超えるべき線）      12.99
XGBoost                13.56
近傍5件の価格中央値             30.86 

trees − all（弱い証拠を外した効果）: -0.744 万円 / 95%区間 [-1.153, -0.357] / 良い確率 100.0%  ← 有意
trees − LightGBM（S6 の受け入れ基準）: -0.139 万円 / 95%区間 [-1.012, +0.728] / 良い確率 61.3%  ← 0 をまたぐ＝有意でない


## 3. どの行で LLM を信じるか — 信号を探した

行ごとに LLM と LightGBM の良いほうを選べたら MAE 10.37（**オラクル**＝実現不可能な上限）。
LightGBM の 12.99 より 2.6 も良いので、見分ける信号があれば伸びしろは大きい。
LLM の自己申告 confidence を含む 7 通りを、上位 r% だけ LLM を採用して比べた。

In [4]:
sig = read("routing_signals.csv").set_index("Unnamed: 0").rename_axis(None)
print(sig.round(2).to_string(), "\n")

best = sig.stack().idxmin()
print(f"最良: 「{best[0]}」を {best[1]} 採用 → MAE {sig.stack().min():.2f}"
      f"（LightGBM 12.99 / オラクル 10.37）")
print("→ ただし信号も採用率も同じ 300 行の上で選んでいるので、この数字は楽観側に偏る。")
print("→ 検定すると最良でも 95%区間が 0 をまたぐ。「有望な候補が見えた」以上は言えない。")

                              0%    10%    20%    30%    50%    75%   100%
LLM の自己申告 confidence（高い順）  12.99  12.95  12.90  12.87  12.86  12.88  12.85
木2つの食い違い |LGBM−XGB|（大きい順）  12.99  12.72  12.84  12.87  12.92  12.96  12.85
木2つの食い違い（小さい順）             12.99  13.02  12.91  12.95  12.91  13.09  12.85
近傍1位の類似度（低い順）              12.99  13.07  13.13  13.08  13.08  12.92  12.85
近傍1位の類似度（高い順）              12.99  12.77  12.81  12.95  12.75  12.69  12.85
LLM が動かした量（大きい順）           12.99  12.91  12.85  12.78  12.82  12.80  12.85
LLM が動かした量（小さい順）           12.99  12.98  13.02  13.03  13.02  13.15  12.85 

最良: 「近傍1位の類似度（高い順）」を 75% 採用 → MAE 12.69（LightGBM 12.99 / オラクル 10.37）
→ ただし信号も採用率も同じ 300 行の上で選んでいるので、この数字は楽観側に偏る。
→ 検定すると最良でも 95%区間が 0 をまたぐ。「有望な候補が見えた」以上は言えない。


## 4. 結論と、東京で決めること

1. **機能A のフォールバックは成功。** 回した行は全問正解、1行 $0.003〜0.005。
   信頼度ルーティング（PRD §6.3）の前提はこの課題では成立している。
2. **設計書の「複数モデルの出力を全部渡す」は実測では間違いだった。**
   弱いモデル（近傍中央値）を混ぜると LLM が引きずられ、MAE が 0.74 悪化する（有意）。
   **証拠は質で選別して渡す必要がある。** 今日いちばん確度の高い発見。
3. **選別しても LightGBM を有意には超えていない。** 受け入れ基準 S6 は未達。
   300 行では ±0.8 万円が判別できないので、行数を増やすか、機能B のゴールを
   精度以外（来歴・説明可能性）に置き直すかの判断が要る。
4. **confidence は機能A では効き、機能B では効かない。**
   「テキストに答えが書いてある分類」と「数値証拠を重み付ける回帰」で
   LLM の得意不得意が分かれている可能性。
5. **まだ単一車種（シエンタ）でしか測っていない。** Craigslist 側は費用見積り（dry-run）止まり。
   過去に何度も「単一車種では差が出ない」（PRD §2.2-b）を踏んでいるので、**次はここ**。